#  ANÁLISIS AGRÍCOLA - PERÚ 2025
###  Herramientas: Pandas, Matplotlib, Seaborn

In [1]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from pathlib import Path

### Carpeta donde se guardarán los gráficos

In [3]:
CARPETA_GRAFICOS = "graficos"

In [4]:
Path(CARPETA_GRAFICOS).mkdir(exist_ok=True)

### PASO 1: CARGAR Y CONOCER EL DATASET

In [5]:
# Leemos el archivo CSV con pandas
df = pd.read_csv("AgriculturaPeru.csv", sep=";")

In [6]:
# ¿Cuántas filas y columnas tiene?
print(f"\nFilas:    {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")


Filas:    2500
Columnas: 12


In [7]:
# Vemos las primeras 5 filas
print("\nPrimeras 5 filas:")
print(df.head())


Primeras 5 filas:
  ID_Produccion       Fecha    Cultivo       Región  Área_sembrada  \
0     PROD-0798  01/01/2025      Arroz  La Libertad           4151   
1     PROD-2015  01/01/2025    Alfalfa     Arequipa           1715   
2     PROD-0509  01/01/2025  Espárrago         Lima           2346   
3     PROD-1463  01/01/2025    Alfalfa     Arequipa           1634   
4     PROD-2190  01/01/2025    Cebolla     Arequipa           1191   

   Producción_toneladas  Costo_produccion  Precio_venta_por_tonelada  \
0               40161.0        21372456.0                   109008.0   
1               47978.0         4889525.0                    20966.0   
2               20877.0        29925141.0                   306329.0   
3               43256.0         4149152.0                    24638.0   
4               27926.0         6159125.0                    40854.0   

   Ingresos_totales Tipo_clima Uso_fertilizante Tipo_riego  
0        43778703.0   Templado               Sí  Aspersión  
1    

In [8]:
# ¿Qué tipo de dato tiene cada columna?
print("\nTipos de datos:")
print(df.dtypes)


Tipos de datos:
ID_Produccion                 object
Fecha                         object
Cultivo                       object
Región                        object
Área_sembrada                  int64
Producción_toneladas         float64
Costo_produccion             float64
Precio_venta_por_tonelada    float64
Ingresos_totales             float64
Tipo_clima                    object
Uso_fertilizante              object
Tipo_riego                    object
dtype: object


In [9]:
# ¿Hay valores vacíos (nulos)?
print("\nValores nulos por columna:")
print(df.isnull().sum())


Valores nulos por columna:
ID_Produccion                  0
Fecha                          0
Cultivo                        0
Región                         0
Área_sembrada                  0
Producción_toneladas          83
Costo_produccion             116
Precio_venta_por_tonelada     76
Ingresos_totales              83
Tipo_clima                    52
Uso_fertilizante              88
Tipo_riego                     0
dtype: int64


### PASO 2: LIMPIAR LOS DATOS

In [10]:
# Guardamos el total de filas antes de limpiar
total_antes = len(df)

In [11]:
# Eliminamos filas donde faltan datos importantes
df = df.dropna(subset=["Producción_toneladas",
                        "Precio_venta_por_tonelada",
                        "Ingresos_totales"])
 
total_despues = len(df)
eliminadas = total_antes - total_despues
print(f"\nFilas eliminadas por datos incompletos: {eliminadas}")
print(f"Filas que quedan para el análisis:      {total_despues}")



Filas eliminadas por datos incompletos: 156
Filas que quedan para el análisis:      2344


In [12]:
# Convertimos la columna Fecha a formato fecha
df["Fecha"] = pd.to_datetime(df["Fecha"], format="%d/%m/%Y")

In [13]:
# Creamos una columna nueva con el número de mes (1=enero, 12=diciembre)
df["Mes"] = df["Fecha"].dt.month

In [14]:
# Creamos una columna de rentabilidad = ingresos - costo
df["Rentabilidad"] = df["Ingresos_totales"] - df["Costo_produccion"]

In [15]:
print("\nColumnas nuevas creadas: Mes, Rentabilidad")
print("\nEstadísticas generales:")
print(df[["Área_sembrada", "Producción_toneladas",
          "Ingresos_totales", "Rentabilidad"]].describe().round(0))


Columnas nuevas creadas: Mes, Rentabilidad

Estadísticas generales:
       Área_sembrada  Producción_toneladas  Ingresos_totales  Rentabilidad
count         2344.0                2344.0      2.344000e+03  2.235000e+03
mean          2734.0               31202.0      2.864703e+07  1.502638e+07
std           4282.0               50780.0      2.309414e+08  2.351877e+08
min              8.0                  19.0      4.107900e+04 -6.345517e+07
25%            895.0                4684.0      5.092269e+06 -9.984070e+05
50%           2258.0               15158.0      1.283098e+07  2.495832e+06
75%           3635.0               36215.0      2.868401e+07  1.097364e+07
max          49189.0              550551.0      1.073573e+10  1.070261e+10


### Analisis de Negocio

In [16]:
# Cuál es el cultivo más producido? 
produccion_por_cultivo = df.groupby("Cultivo")["Producción_toneladas"].sum()
produccion_por_cultivo = produccion_por_cultivo.sort_values(ascending=False)

print("\n1. Producción total por cultivo (toneladas):")
print(produccion_por_cultivo.round(0).to_string())


1. Producción total por cultivo (toneladas):
Cultivo
Caña de azúcar    19448872.0
Papa              13228579.0
Maíz               6833427.0
Camote             5728112.0
Yuca               5183761.0
Alfalfa            4699248.0
Arroz              4231896.0
Cebolla            3543014.0
Espárrago          2849333.0
Plátano            2373059.0
Mango              1640486.0
Limón              1243517.0
Quinua             1159340.0
Kiwicha             456094.0
Trigo               371961.0
Cañihua             146323.0


In [17]:
# ¿Qué región genera más ingresos? 
ingresos_por_region = df.groupby("Región")["Ingresos_totales"].sum()
ingresos_por_region = ingresos_por_region.sort_values(ascending=False)
 
print("\n2. Ingresos totales por región (soles):")
print(ingresos_por_region.round(0).to_string())


2. Ingresos totales por región (soles):
Región
Lambayeque     1.466418e+10
Lima           1.005283e+10
La Libertad    8.543910e+09
Piura          6.227159e+09
Arequipa       5.331972e+09
Junín          5.314929e+09
Cusco          4.893557e+09
Puno           4.698187e+09
Cajamarca      3.793544e+09
Loreto         3.628366e+09


In [18]:
# ¿Cuántos registros son rentables?
rentables     = (df["Rentabilidad"] > 0).sum()
no_rentables  = (df["Rentabilidad"] <= 0).sum()
pct_rentables = rentables / len(df) * 100
 
print(f"\n4. Registros rentables:     {rentables} ({pct_rentables:.1f}%)")
print(f"   Registros no rentables:  {no_rentables}")


4. Registros rentables:     1494 (63.7%)
   Registros no rentables:  741


### PASO 4: VISUALIZACIONES

In [19]:
# Paleta de colores 
COLOR_PRINCIPAL = "#2563EB"   # azul
COLOR_VERDE     = "#16A34A"   # verde
COLOR_ROJO      = "#DC2626"   # rojo
COLOR_NARANJA   = "#D97706"   # naranja
FONDO           = "#F8FAFC"

In [20]:
# GRÁFICO 1: Producción total por cultivo
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(FONDO)
ax.set_facecolor("white")

# Ordenamos de mayor a menor
cultivos  = produccion_por_cultivo.index.tolist()
valores   = (produccion_por_cultivo.values / 1000).tolist()   # en miles
 
ax.bar(cultivos, valores, color=COLOR_PRINCIPAL, alpha=0.85)

<BarContainer object of 16 artists>

In [21]:
# Títulos y etiquetas
ax.set_title("Producción total por cultivo (miles de toneladas)",
             fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("Cultivo")
ax.set_ylabel("Miles de toneladas")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", color="#E2E8F0", linewidth=0.7)
ax.set_axisbelow(True)

In [22]:
plt.tight_layout()
plt.savefig(f"{CARPETA_GRAFICOS}/01_produccion_por_cultivo.png", dpi=150)
plt.close()

In [23]:
#GRÁFICO 2: Ingresos por región
fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor(FONDO)
ax.set_facecolor("white")
 
regiones = ingresos_por_region.index.tolist()
ingresos = (ingresos_por_region.values / 1_000_000).tolist()  # en millones
 
ax.barh(regiones, ingresos, color=COLOR_VERDE, alpha=0.85)
 
ax.set_title("Ingresos totales por región (millones de soles)",
             fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("Millones de S/")
ax.grid(axis="x", color="#E2E8F0", linewidth=0.7)
ax.set_axisbelow(True)

In [24]:
plt.tight_layout()
plt.savefig(f"{CARPETA_GRAFICOS}/02_ingresos_por_region.png", dpi=150)
plt.close()

### GRÁFICO 3: Producción mensual

In [25]:
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor(FONDO)
ax.set_facecolor("white")
 
produccion_mes = df.groupby("Mes")["Producción_toneladas"].sum() / 1000
 
meses_nombres = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
                 "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
 
ax.plot(produccion_mes.index, produccion_mes.values,
        marker="o", linewidth=2, color=COLOR_PRINCIPAL,
        markersize=6, markerfacecolor="white", markeredgewidth=2)
 
ax.fill_between(produccion_mes.index, produccion_mes.values,
                alpha=0.12, color=COLOR_PRINCIPAL)

In [26]:
ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses_nombres)
ax.set_title("Producción mensual durante el año 2025 (miles de toneladas)",
             fontsize=13, fontweight="bold", pad=15)
ax.set_ylabel("Miles de toneladas")
ax.grid(color="#E2E8F0", linewidth=0.7)
ax.set_axisbelow(True)
 
plt.tight_layout()
plt.savefig(f"{CARPETA_GRAFICOS}/03_produccion_mensual.png", dpi=150)
plt.close()

### GRÁFICO 4: Comparación riego vs producción

In [27]:
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor(FONDO)
ax.set_facecolor("white")
 
prod_riego = df.groupby("Tipo_riego")["Producción_toneladas"].mean()
 
colores_riego = [COLOR_PRINCIPAL, COLOR_VERDE, COLOR_NARANJA]
ax.bar(prod_riego.index, prod_riego.values,
       color=colores_riego, alpha=0.85, width=0.5)
 
ax.set_title("Producción promedio según tipo de riego (toneladas)",
             fontsize=13, fontweight="bold", pad=15)
ax.set_ylabel("Toneladas promedio")
ax.grid(axis="y", color="#E2E8F0", linewidth=0.7)
ax.set_axisbelow(True)
 
plt.tight_layout()
plt.savefig(f"{CARPETA_GRAFICOS}/04_produccion_por_riego.png", dpi=150)
plt.close()

### GRÁFICO 5: Rentables vs No rentables

In [28]:
fig, ax = plt.subplots(figsize=(6, 5))
fig.patch.set_facecolor(FONDO)
ax.set_facecolor(FONDO)
 
etiquetas = ["Rentables", "No rentables"]
cantidades = [rentables, no_rentables]
colores    = [COLOR_VERDE, COLOR_ROJO]
 
ax.pie(cantidades, labels=etiquetas, colors=colores,
       autopct="%1.1f%%", startangle=90,
       wedgeprops=dict(edgecolor="white", linewidth=2),
       textprops={"fontsize": 11})
 
ax.set_title("¿Cuántos registros son rentables?",
             fontsize=13, fontweight="bold", pad=15)
 
plt.tight_layout()
plt.savefig(f"{CARPETA_GRAFICOS}/05_rentabilidad.png", dpi=150)
plt.close()

### Guardar Dataset Limpio

In [29]:
df.to_csv('AnalisisAgricolaLimpio.csv', index=False)